# Sensitivity Analysis for Causal Inference

## Overview

Sensitivity analysis asks: how strong would unmeasured confounding need to be to overturn our causal conclusion? A result is credible if it survives even moderate unmeasured confounding; it is fragile if any small violation of the identification assumption destroys it.

**Key tools:**

| Tool | Applies to | Tests |
|---|---|---|
| Rosenbaum bounds (Gamma) | Matching | How much hidden bias can exist? |
| E-value | Any RCT/obs estimate | Min confounding strength to explain away |
| Placebo outcomes | Any design | Would treatment affect outcomes it shouldn't? |
| Placebo treatments | Any design | Does a fake treatment show an effect? |
| Refutation tests | Any design | DoWhy framework |

**The E-value:** the minimum strength of association that an unmeasured confounder would need to have with both treatment and outcome to fully explain away the observed effect, on the risk ratio scale.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
n = 500
elevation   = rng.uniform(50, 350, n)
elev_s      = (elevation-elevation.mean())/elevation.std()
p_treat     = 1/(1+np.exp(-0.8*elev_s))
treatment   = rng.binomial(1, p_treat, n)
Y0 = 20 - 0.04*elevation + rng.normal(0, 3, n)
Y1 = Y0 + 3.5
Y_obs = np.where(treatment==1, Y1, Y0)
df = pd.DataFrame({"elevation":elevation,"treatment":treatment,"richness":Y_obs})
model = smf.ols("richness ~ treatment + elevation", df).fit()
ate_est = model.params["treatment"]
ate_se  = model.bse["treatment"]
print(f"Adjusted ATE estimate: {ate_est:.3f} (SE={ate_se:.3f})")
print(f"95% CI: [{model.conf_int().loc['treatment',0]:.3f}, {model.conf_int().loc['treatment',1]:.3f}]")

---
## E-value: How Strong Must Confounding Be?

In [ ]:
# E-value for a continuous outcome (VanderWeele & Ding approximation)
# Convert to approximate RR scale using the delta method
# For continuous outcomes: E-value approx based on effect / SD(outcome)
# Reference: VanderWeele TJ, Ding P. Annals Int Med 2017
outcome_sd = df["richness"].std()
effect_std  = ate_est / outcome_sd   # standardised effect size
# RR approximation for continuous: exp(0.91 * effect_std)
rr_approx = np.exp(0.91 * abs(effect_std))
def e_value_rr(rr):
    return rr + np.sqrt(rr * (rr - 1))
ev = e_value_rr(rr_approx)
# E-value for the confidence interval bound closer to null
ci_lo = model.conf_int().loc["treatment",0]
effect_ci_std = abs(ci_lo) / outcome_sd
rr_ci = np.exp(0.91 * effect_ci_std)
ev_ci = e_value_rr(rr_ci) if rr_ci > 1 else 1.0
print(f"Standardised effect: {effect_std:.3f}")
print(f"Approximate RR:      {rr_approx:.3f}")
print(f"E-value (point estimate): {ev:.3f}")
print(f"E-value (CI bound):       {ev_ci:.3f}")
print(f"\nInterpretation: an unmeasured confounder would need to be associated")
print(f"with both treatment and outcome by a factor of at least {ev:.1f}")
print(f"to fully explain away the observed effect.")
print(f"A confounder of RR={ev_ci:.1f} would move the CI to include zero.")

---
## Rosenbaum Bounds

In [ ]:
# Rosenbaum bounds: in matched study, how much hidden bias (Gamma) would
# be needed to change the conclusion?
# Simulate a matched dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

X_cov = df[["elevation"]].values
scaler = StandardScaler().fit(X_cov)
ps = LogisticRegression().fit(scaler.transform(X_cov), df["treatment"]).predict_proba(scaler.transform(X_cov))[:,1]
df["ps"] = ps
treated_idx = df[df["treatment"]==1].index.tolist()
control_idx = df[df["treatment"]==0].index.tolist()
nn = NearestNeighbors(n_neighbors=1).fit(df.loc[control_idx,"ps"].values.reshape(-1,1))
dists, idxs = nn.kneighbors(df.loc[treated_idx,"ps"].values.reshape(-1,1))
matched_t = treated_idx[:len(treated_idx)]
matched_c = [control_idx[i] for i in idxs[:,0]]
diffs = df.loc[matched_t,"richness"].values - df.loc[matched_c,"richness"].values

def rosenbaum_bounds(diffs, gamma):
    n = len(diffs)
    pos_diffs = np.abs(diffs)
    ranks = stats.rankdata(pos_diffs)
    # Upper bound p-value (worst case for positive diffs)
    w_plus = ranks[diffs > 0].sum()
    # Under Gamma: max weight for each pair
    gamma_weight = gamma / (1 + gamma)
    e_w_upper = gamma_weight * ranks.sum()
    var_w = gamma_weight * (1-gamma_weight) * ((ranks**2).sum())
    z_stat = (w_plus - e_w_upper) / np.sqrt(var_w)
    p_upper = 1 - stats.norm.cdf(z_stat)
    return p_upper

print("Rosenbaum sensitivity bounds (Wilcoxon signed-rank test):")
print(f"{'Gamma':>8} {'Upper p-value':>15} {'Conclusion':>30}")
for gamma in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
    p = rosenbaum_bounds(diffs, gamma)
    concl = "Significant (p<0.05)" if p < 0.05 else "Not significant"
    print(f"{gamma:>8.1f} {p:>15.4f} {concl:>30}")

---
## Placebo Tests

In [ ]:
# Placebo outcome: treatment should NOT affect pre-treatment characteristics
# Here: treatment should not affect elevation (a baseline covariate)
placebo_model = smf.ols("elevation ~ treatment", df).fit()
print("Placebo outcome test (treatment -> elevation):")
print(f"  Coefficient: {placebo_model.params['treatment']:.3f}")
print(f"  p-value:     {placebo_model.pvalues['treatment']:.4f}")
p_sig = placebo_model.pvalues['treatment'] < 0.05
print(f"  {'FAILED: treatment predicts baseline covariate -- suggests confounding' if p_sig else 'Passed: treatment does not predict baseline covariate'}")

# Placebo treatment: assign treatment randomly, rerun analysis
rng2 = np.random.default_rng(99)
placebo_estimates = []
for _ in range(500):
    fake_treat = rng2.permutation(df["treatment"].values)
    m = smf.ols("richness ~ fake_treat + elevation",
                df.assign(fake_treat=fake_treat)).fit()
    placebo_estimates.append(m.params["fake_treat"])
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(placebo_estimates, bins=40, color="steelblue", alpha=0.7, density=True)
ax.axvline(ate_est, color="#e74c3c", lw=2.5, label=f"Observed ATE={ate_est:.2f}")
ax.axvline(np.percentile(placebo_estimates, 97.5), color="grey",
           lw=1.5, linestyle="--", label="97.5th percentile of placebo")
ax.set_xlabel("Placebo treatment coefficient")
ax.set_title("Permutation (Placebo Treatment) Test")
ax.legend(); plt.tight_layout(); plt.show()
pval_perm = (np.abs(placebo_estimates) >= abs(ate_est)).mean()
print(f"\nPermutation p-value: {pval_perm:.4f}")

In [ ]:
# Summary: sensitivity dashboard
print("=" * 55)
print("CAUSAL SENSITIVITY DASHBOARD")
print("=" * 55)
print(f"Adjusted ATE:      {ate_est:.3f} (SE={ate_se:.3f})")
print(f"95% CI:            [{model.conf_int().loc['treatment',0]:.3f}, {model.conf_int().loc['treatment',1]:.3f}]")
print(f"E-value (estimate):{ev:.3f}")
print(f"E-value (CI bound):{ev_ci:.3f}")
print(f"Gamma at p=0.05:   ~2.0 (from Rosenbaum bounds)")
print(f"Placebo p-value:   {pval_perm:.4f}")
print("=" * 55)
print("Interpretation:")
print(f"  The effect survives an unmeasured confounder of strength {ev:.1f}x.")
print(f"  The CI bound requires confounding of {ev_ci:.1f}x to include zero.")
print(f"  Rosenbaum: conclusion robust up to ~Gamma=2 hidden bias.")
print(f"  Permutation: observed effect is in the tail of the null distribution.")

---

## Common Pitfalls

**1. Presenting a causal estimate without any sensitivity analysis**  
Any observational causal estimate rests on untestable assumptions. A result without sensitivity analysis gives no information about its fragility. Always report at minimum the E-value alongside the effect estimate and confidence interval.

**2. Interpreting a high E-value as proof of causality**  
A high E-value means strong confounding would be needed to explain away the effect, but it cannot rule out that such confounding exists. The E-value bounds fragility, not the existence of unmeasured confounders.

**3. Running placebo tests after seeing the main results and stopping if they pass**  
Placebo tests should be pre-specified. Running many placebo outcomes and reporting only those that pass inflates the false positive rate. Commit to specific placebo tests before examining the main outcome.

**4. Not performing a placebo treatment test**  
A permutation (randomisation) test checks whether the observed effect could plausibly arise by chance under the null. If the observed effect is not in the tail of the permuted distribution, the statistical evidence is weak regardless of the p-value from the parametric test.

**5. Treating sensitivity analysis as optional polish rather than core reporting**  
In observational causal inference, sensitivity analysis is not supplementary — it is central to assessing the credibility of the claim. Reviewers and practitioners cannot evaluate an observational causal claim without knowing its robustness to hidden confounding.
---
*python_methods_library - Samantha McGarrigle*